# Day 01: House Price Prediction

In [98]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [99]:
df=pd.read_csv("/Users/nidagallusandeep/Downloads/house_price_prediction_20000_rows_12_columns.csv")

In [100]:
df.head()

,area_sqft,bedrooms,bathrooms,house_age_years,location,property_type,parking_spaces,floor_number,distance_to_city_km,nearby_schools,nearby_hospitals,price_inr
0,3900,6.0,5,19,City,Independent House,0,1,4.8,3,1,49998606
1,1174,1.0,1,1,Rural,Independent House,1,1,8.1,7,0,14636002
2,557,1.0,1,7,City,Apartment,1,18,9.9,1,1,9119395
3,3042,6.0,6,1,City,Independent House,2,1,8.1,5,2,40832144
4,1930,5.0,5,31,City,Apartment,2,3,2.6,3,1,25906534


In [101]:
df.shape

(20000, 12)

In [102]:
df.columns

Index(['area_sqft', 'bedrooms', 'bathrooms', 'house_age_years', 'location',
       'property_type', 'parking_spaces', 'floor_number',
       'distance_to_city_km', 'nearby_schools', 'nearby_hospitals',
       'price_inr'],
      dtype='object')

In [103]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   area_sqft            20000 non-null  int64  
 1   bedrooms             19900 non-null  float64
 2   bathrooms            20000 non-null  int64  
 3   house_age_years      20000 non-null  int64  
 4   location             19920 non-null  object 
 5   property_type        19950 non-null  object 
 6   parking_spaces       20000 non-null  int64  
 7   floor_number         20000 non-null  int64  
 8   distance_to_city_km  19900 non-null  float64
 9   nearby_schools       20000 non-null  int64  
 10  nearby_hospitals     20000 non-null  int64  
 11  price_inr            20000 non-null  int64  
dtypes: float64(2), int64(8), object(2)
memory usage: 1.8+ MB


In [104]:
df.describe()

,area_sqft,bedrooms,bathrooms,house_age_years,parking_spaces,floor_number,distance_to_city_km,nearby_schools,nearby_hospitals,price_inr
count,20000.000000,19900.000000,20000.000000,20000.000000,20000.000000,20000.000000,19900.000000,20000.000000,20000.000000,2.000000e+04
mean,2480.764750,3.836281,3.709800,19.962300,1.180800,7.119000,9.868879,3.002300,1.491500,3.172888e+07
std,1170.897869,1.884613,1.757168,11.818442,1.066428,6.443773,6.627047,1.721494,1.219878,1.404843e+07
min,450.000000,1.000000,1.000000,0.000000,0.000000,1.000000,0.500000,0.000000,0.000000,4.098806e+06
25%,1462.000000,2.000000,2.000000,10.000000,0.000000,1.000000,5.000000,2.000000,1.000000,1.960983e+07
50%,2479.000000,4.000000,4.000000,20.000000,1.000000,5.000000,8.400000,3.000000,1.000000,3.173778e+07
75%,3491.000000,5.000000,5.000000,30.000000,2.000000,13.000000,13.100000,4.000000,2.000000,4.380197e+07
max,4500.000000,7.000000,6.000000,40.000000,4.000000,20.000000,40.000000,11.000000,8.000000,6.030249e+07


In [127]:
Q1 = df["price_inr"].quantile(0.25)
Q3 = df["price_inr"].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
outliers = df[(df["price_inr"] < lower) | (df["price_inr"] > upper)]
print("Number of outliers:", len(outliers))
print(outliers[["price_inr"]])

Number of outliers: 0
Empty DataFrame
Columns: [price_inr]
Index: []


In [128]:
df.isnull().sum()

area_sqft              0
bedrooms               0
bathrooms              0
house_age_years        0
location               0
property_type          0
parking_spaces         0
floor_number           0
distance_to_city_km    0
nearby_schools         0
nearby_hospitals       0
price_inr              0
dtype: int64

In [129]:
df["bedrooms"]=df["bedrooms"].fillna(df["bedrooms"].mean())
df["location"]=df["location"].fillna(df["location"].mode()[0])
df["property_type"]=df["property_type"].fillna(df["property_type"].mode(0))
df["distance_to_city_km"]=df["distance_to_city_km"].fillna(df["distance_to_city_km"].mode()[0])

In [130]:
df["property_type"]=df["property_type"].fillna(df["property_type"].mode()[0])

In [131]:
df.isnull().sum()

area_sqft              0
bedrooms               0
bathrooms              0
house_age_years        0
location               0
property_type          0
parking_spaces         0
floor_number           0
distance_to_city_km    0
nearby_schools         0
nearby_hospitals       0
price_inr              0
dtype: int64

In [132]:
df.duplicated().sum()

np.int64(0)

# Separate features and target


In [133]:
X=df.drop(columns="price_inr",axis=1)
y=df["price_inr"]

In [134]:
numeric_features = [
    "area_sqft",
    "bedrooms",
    "bathrooms",
    "house_age_years",
    "parking_spaces",
    "floor_number",
    "distance_to_city_km",
    "nearby_schools",
    "nearby_hospitals"]

categorical_features = [
    "location",
    "property_type"]

In [135]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=42)

In [136]:
# 5. Handle missing values
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))])

In [137]:
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))])

In [138]:
# 6. Apply encoding using ColumnTransformer
preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)])

In [139]:
# 7. Build baseline model
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])
# 8. Train baseline model
model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](11,)","['area_sqft','bedrooms','bathrooms',...,'distance_to_city_km', 'nearby_schools','nearby_hospitals']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,11
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automaticall

In [140]:
# 9. Baseline predictions
y_pred =model.predict(X_test)

In [141]:
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score

In [142]:
mae=mean_absolute_error(y_test,y_pred)
print("MAE:",mae)
mse=mean_squared_error(y_test,y_pred)
print("MSE:",mse)
r2=r2_score(y_test,y_pred)
print("R2_Score:",r2)

MAE: 530169.6825031717
MSE: 451389721389.49036
R2_Score: 0.9977596358898503


In [143]:
# 10. Hyperparameter tuning
param_grid = {
    "model__fit_intercept": [True, False],
    "model__positive": [True, False]}


In [144]:
grid_search = GridSearchCV(
    estimator=Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression())]),
    param_grid=param_grid,
    scoring="neg_mean_absolute_error",
    cv=5,
    n_jobs=-1,
    verbose=1)

grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 4 candidates, totalling 20 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...egression())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__fit_intercept': [True, False], 'model__positive': [True, False]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_mean_absolute_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_

In [145]:
pred=grid_search.predict(X_test)

In [146]:
mae=mean_absolute_error(y_test,pred)
print("MAE:",mae)
mse=mean_squared_error(y_test,pred)
print("MSE:",mse)
r2=r2_score(y_test,pred)
print("R2_Score:",r2)

MAE: 530169.6825031717
MSE: 451389721389.49036
R2_Score: 0.9977596358898503
